# Step 1: Clone

In [1]:
from __future__ import annotations
import csv, os, re, subprocess, time
from pathlib import Path
from typing import Optional, List

# -----------------------------
# Config (edit as needed)
# -----------------------------
WORK_ROOT    = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
URL_LIST_CSV = WORK_ROOT / "URL_List.csv"
CLONE_ROOT   = WORK_ROOT / "clonesV8.1"
MANIFEST_CSV = WORK_ROOT / "clones_manifestV8.1.csv"

WITH_SUBMODULES = False     # set True if you want submodules initialized/updated
WITH_LFS        = False     # set True if you need Git LFS objects
FETCH_PR_REFS   = True      # set False to skip GitHub PR heads

# Create dirs
WORK_ROOT.mkdir(parents=True, exist_ok=True)
CLONE_ROOT.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Helpers
# -----------------------------
DEFAULT_TIMEOUT = 1800  # 30 minutes for huge repos
GIT_ENV = {"GIT_TERMINAL_PROMPT": "0", "GIT_ASKPASS": ""}

def sh(cmd: List[str], cwd: Optional[Path] = None, check: bool = True,
       capture: bool = True, timeout: Optional[int] = DEFAULT_TIMEOUT) -> subprocess.CompletedProcess:
    env = os.environ.copy()
    env.update(GIT_ENV)
    return subprocess.run(cmd, cwd=cwd, check=check,
                          capture_output=capture, text=True,
                          timeout=timeout, env=env)

def sh_ok(cmd: List[str], cwd: Optional[Path] = None, timeout: Optional[int] = DEFAULT_TIMEOUT) -> str:
    cp = sh(cmd, cwd=cwd, check=True, capture=True, timeout=timeout)
    return cp.stdout

def repo_dir_name_from_url(url: str) -> str:
    u = url.strip()
    # SSH form: git@host:owner/repo(.git)
    m = re.match(r"^[^@]+@([^:]+):([^/]+)/(.+?)(?:\.git)?$", u)
    if m:
        owner, name = m.group(2), m.group(3)
        return f"{owner}__{name}"
    # https://host/owner/repo(.git)
    if "://" in u:
        base = u.split("://", 1)[1]
    else:
        base = u
    parts = [p for p in base.split("/") if p]
    if len(parts) >= 2:
        owner, name = parts[-2], parts[-1].removesuffix(".git")
        return f"{owner}__{name}"
    return base.replace("/", "__").removesuffix(".git")

def ensure_full_clone(url: str, dest_root: Path,
                      with_submodules: bool = False,
                      with_lfs: bool = False,
                      fetch_pr_refs: bool = True) -> Path:
    """
    Ensures a non-shallow clone with full history of all branches & tags.
    Optionally fetches GitHub PR heads to refs/remotes/origin/pr/*,
    initializes submodules, and fetches LFS objects.
    """
    dest_root.mkdir(parents=True, exist_ok=True)
    d = dest_root / repo_dir_name_from_url(url)

    if not (d.exists() and (d / ".git").exists()):
        # no-recurse-submodules avoids submodule cost unless requested later
        sh(["git", "clone", "--no-recurse-submodules", "--tags", url, str(d)],
           capture=False)
    else:
        # If repo exists, ensure origin URL is correct
        sh(["git", "remote", "set-url", "origin", url], cwd=d, check=False)

    # If shallow, unshallow
    cp = sh(["git", "rev-parse", "--is-shallow-repository"], cwd=d, check=False)
    if cp.returncode == 0 and cp.stdout.strip() == "true":
        sh(["git", "fetch", "--unshallow", "--tags"], cwd=d, capture=False)
    else:
        # Make sure we have tags even if already full
        sh(["git", "fetch", "--tags"], cwd=d, check=False, capture=False)

    # Fetch all branches under refs/heads/* and prune deleted ones
    sh(["git", "fetch", "origin", "--prune", "--tags",
        "+refs/heads/*:refs/remotes/origin/*", "--quiet"], cwd=d, check=False, capture=False)

    # Best-effort PR refs (GitHub); harmless if not present
    if fetch_pr_refs:
        sh(["git", "fetch", "origin",
            "+refs/pull/*/head:refs/remotes/origin/pr/*", "--quiet"], cwd=d, check=False, capture=False)

    # Optional: submodules
    if with_submodules:
        sh(["git", "submodule", "update", "--init", "--recursive"], cwd=d, capture=False)

    # Optional: LFS
    if with_lfs:
        # If git-lfs isn't installed, these will fail harmlessly due to check=False
        sh(["git", "lfs", "install"], cwd=d, check=False, capture=False)
        sh(["git", "lfs", "fetch", "--all"], cwd=d, check=False, capture=False)
        sh(["git", "lfs", "checkout"], cwd=d, check=False, capture=False)

    return d

def get_total_commits(repo_dir: Path) -> int:
    # Count across all refs reachable in the repository
    cp = sh(["git", "rev-list", "--all", "--count"], cwd=repo_dir, check=False)
    try:
        return int((cp.stdout or "0").strip() or "0")
    except Exception:
        return 0

# -----------------------------
# Main: FULL CLONE ONLY
# -----------------------------
assert URL_LIST_CSV.exists(), f"CSV not found: {URL_LIST_CSV}"

rows, ok, fail = [], 0, 0
with URL_LIST_CSV.open(newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        url = (row.get("repo_url") or "").strip()
        if not url:
            continue
        t0 = time.time()
        rec = {
            "repo_url": url,
            "dir": None,
            "status": "unknown",
            "seconds": None,
            "total_commits": None,
            "error": ""
        }
        try:
            d = ensure_full_clone(url, CLONE_ROOT,
                                  with_submodules=WITH_SUBMODULES,
                                  with_lfs=WITH_LFS,
                                  fetch_pr_refs=FETCH_PR_REFS)
            rec["dir"] = str(d)
            rec["total_commits"] = get_total_commits(d)
            rec["status"] = "ok"
            ok += 1
        except subprocess.CalledProcessError as e:
            rec["status"] = "error"
            rec["error"]  = (e.stderr or e.stdout or str(e)).strip()[:2000]
            fail += 1
        except Exception as e:
            rec["status"] = "error"
            rec["error"]  = str(e)[:2000]
            fail += 1

        rec["seconds"] = round(time.time() - t0, 2)
        rows.append(rec)
        print(f"[{rec['status']}] {url} -> {rec['dir']} ({rec['seconds']}s)  commits={rec['total_commits']}")

# Write manifest
MANIFEST_CSV.parent.mkdir(parents=True, exist_ok=True)
fieldnames = ["repo_url","dir","status","seconds","total_commits","error"]
with MANIFEST_CSV.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(rows)

print(f"\nDone. OK={ok}, FAIL={fail}. Manifest: {MANIFEST_CSV}")


[ok] https://github.com/Rajawali/Rajawali -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clonesV8.1\Rajawali__Rajawali (19.6s)  commits=3065
[ok] https://github.com/splitwise/TokenAutoComplete -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clonesV8.1\splitwise__TokenAutoComplete (1.65s)  commits=428
[ok] https://github.com/OpnTec/bodyapps-android -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clonesV8.1\OpnTec__bodyapps-android (2.02s)  commits=160
[ok] https://github.com/Stuart-campbell/RushOrm -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clonesV8.1\Stuart-campbell__RushOrm (1.48s)  commits=216
[ok] https://github.com/kost/NetworkMapper -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clonesV8.1\kost__NetworkMapper (1.09s)  commits=71
[ok] https://github.com/Redgram/redgram-for-reddit -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clonesV8.1\Redgram__redgram-for-reddit (1.95s)  commits=328
[ok] https://githu

## Step 2: Mining 

In [5]:
# %% [markdown]
# Step 2 (FAST, THREADS): miner (with INTENT_GMD) — Commit-diff & blob-cached
# - Reads repos from: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones
# - Writes JSON to:   C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\mine
# - Detects styles: DIY, GENERIC (ReactiveCircus/Malinskiy), GMD (Gradle DSL), INTENT_GMD (YAML-only intent)
# - Major speedups:
#     * Scan only commits that changed YAML/Gradle (git rev-list pathspec + diff-tree)
#     * Read blobs by SHA with LRU cache (git cat-file -p) instead of show commit:path each time
#     * Optional: skip merge commits
#     * Size gate blobs before reading
#     * Parallelize across repos using THREADS (I/O-bound workload)
#     * Resume: skip repos that already have output JSON
# - No external dependencies.

# %%
import json
import os
import re
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor as PoolExecutor, as_completed
from datetime import datetime
from functools import lru_cache
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple
import time
import random

# ---------------------------
# Config (edit as needed)
# ---------------------------
BASE_FOLDER = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
INPUT_CLONES = BASE_FOLDER / "clones"
OUTPUT_MINE = BASE_FOLDER / "mine"

# Git cutoff (ISO). Git understands "--before".
CUTOFF_DEFAULT = "2025-08-10T23:59:59"

# Limit number of repos for testing; 0 = no limit
MAX_REPOS = 0

# Speed knobs
SKIP_MERGE_COMMITS = True            # Skip merge commits (safe in most timelines)
BLOB_MAX_SIZE = 600_000              # Bytes; skip huge files early
MAX_WORKERS = min(32, (os.cpu_count() or 8) * 2)  # Thread workers; good for I/O
RESUME_IF_EXISTS = True              # Skip repos with an existing output JSON

# Optional: restrict YAML search initially to CI hotspots (set False for full project YAML)
YAML_CI_FIRST = True

# ---------------------------
# File type helpers
# ---------------------------
YAML_EXTS = (".yml", ".yaml")
GRADLE_NAMES = {
    "build.gradle",
    "build.gradle.kts",
    "settings.gradle",
    "settings.gradle.kts",
}
GRADLE_EXTS = (".gradle", ".gradle.kts")

def is_yaml(path: str) -> bool:
    return os.path.splitext(path)[1].lower() in YAML_EXTS

def is_gradle(path: str) -> bool:
    name = os.path.basename(path)
    ext = os.path.splitext(path)[1].lower()
    return name in GRADLE_NAMES or ext in GRADLE_EXTS

# ---------------------------
# Regex heuristics (unchanged semantics)
# ---------------------------
GENERIC_PATTERNS_ALL = [
    re.compile(r"reactivecircus\s*/\s*android[-_]emulator[-_]runner", re.I),
    re.compile(r"malinskiy\s*/\s*action[-_]android", re.I),
    re.compile(r"malinskiy\s*/\s*android[-_]emulator[-_]runner", re.I),
]

DIY_PATTERNS_ALL = [
    re.compile(r"\bemulator(\.exe)?\s*-[A-Za-z]", re.I),
    re.compile(r"\bavdmanager\b", re.I),
    re.compile(r"\bsdkmanager\b", re.I),
    re.compile(r"\badb\b", re.I),
    re.compile(r"\bandroid\s+create\s+avd\b", re.I),
    re.compile(r"\bemulator-headless\b", re.I),
    re.compile(r"system-images;android-\d+;google_apis(;x86_64|;x86)?", re.I),
    re.compile(r"echo\s+no\s*\|\s*avdmanager\s+create\s+avd", re.I),
    re.compile(r"emulator\s+-list-avds", re.I),
]

# True GMD (Gradle DSL in Gradle files)
GMD_PATTERNS_GRADLE = [
    re.compile(r"\btestOptions\s*\{[^}]*managedDevices\s*\{", re.I | re.S),
    re.compile(r"\bmanagedDevices\s*\{", re.I),
    re.compile(r"\bManagedVirtualDevice\b", re.I),
    re.compile(r"\bdevices\s*\{[^}]*\bapi\d+\b", re.I | re.S),
]

# GMD intent (YAML-only)
GMD_PATTERNS_YAML_INTENT = [
    re.compile(r"\bgradle\s+managed\s+devices?\b", re.I),
    re.compile(r"\bmanaged\s+devices?\b", re.I),
    re.compile(r"\b[a-z0-9]+api\d+(Debug|Release)AndroidTest\b", re.I),
    re.compile(r"\bgradlew?(\.bat)?\s+[:\w\-]*[a-z0-9]+api\d+(Debug|Release)AndroidTest\b", re.I),
    re.compile(r"\bgradlew?(\.bat)?\s+[:\w\-]*allDevices(AndroidTest|Test)\b", re.I),
]

# ---------------------------
# Git helpers (fast path + light retry for Windows hiccups)
# ---------------------------
def run_git(repo: Path, args: List[str], text: bool = True, check: bool = True, retries: int = 2) -> subprocess.CompletedProcess:
    last_exc = None
    for attempt in range(retries + 1):
        try:
            return subprocess.run(
                ["git", "-C", str(repo)] + args,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=text,
                check=check,
                encoding="utf-8" if text else None,
                errors="replace" if text else None,
            )
        except subprocess.CalledProcessError as e:
            last_exc = e
            # Some transient errors; jittered backoff
            time.sleep(0.1 * (attempt + 1) + random.random() * 0.1)
        except Exception as e:
            last_exc = e
            time.sleep(0.05)
    # If we get here, bubble up the last error
    if isinstance(last_exc, subprocess.CalledProcessError) and not check:
        return last_exc
    raise last_exc  # re-raise

def list_repos(root: Path) -> List[Path]:
    repos = []
    for p, dirs, _files in os.walk(root):
        pth = Path(p)
        if (pth / ".git").exists():
            repos.append(pth)
            dirs[:] = []  # don't descend into nested repos
    return repos

def is_merge(repo: Path, commit: str) -> bool:
    try:
        parents = run_git(repo, ["show", "-s", "--format=%P", commit]).stdout.strip()
        return len(parents.split()) >= 2
    except Exception:
        return False

def get_commit_date_iso(repo: Path, commit: str) -> Optional[str]:
    try:
        out = run_git(repo, ["show", "-s", "--format=%cI", commit]).stdout.strip()
        return out or None
    except Exception:
        return None

def _pathspecs_yaml() -> List[str]:
    if not YAML_CI_FIRST:
        return [":(glob)**/*.yml", ":(glob)**/*.yaml"]
    # CI hotspots plus common roots
    return [
        ".github/workflows", ".gitlab-ci.yml", "azure-pipelines.yml",
        ".circleci/config.yml", ".bitrise.yml",
        ":(glob)**/*.yml", ":(glob)**/*.yaml",
    ]

def _pathspecs_gradle() -> List[str]:
    return [
        "build.gradle", "build.gradle.kts", "settings.gradle", "settings.gradle.kts",
        ":(glob)**/*.gradle", ":(glob)**/*.gradle.kts",
    ]

def commits_touching_relevant(repo: Path, cutoff_iso: str) -> List[str]:
    pathspecs = _pathspecs_yaml() + _pathspecs_gradle()
    args = ["rev-list", "--reverse", "--before", cutoff_iso, "--all", "--"]
    try:
        out = run_git(repo, args + pathspecs).stdout
        commits = [c for c in out.splitlines() if c.strip()]
        return commits
    except Exception:
        return []

def changed_relevant_files(repo: Path, commit: str) -> List[str]:
    """
    Return paths of YAML/Gradle files changed in this commit.
    """
    try:
        out = run_git(repo, ["diff-tree", "-r", "--no-commit-id", "-z", commit]).stdout
    except Exception:
        return []
    items = out.split("\x00")
    paths: List[str] = []
    # Name-status output is NUL separated; entries come as STATUS + PATH (plus for renames)
    i = 0
    while i < len(items):
        entry = items[i]
        i += 1
        if not entry:
            continue
        status = entry.split("\t")[0] if "\t" in entry else entry  # "A", "M", "R100", etc.
        # For renames, next two fields are old and new; for others, next is path
        if status.startswith("R") or status.startswith("C"):
            if i + 1 >= len(items):
                break
            old_path = items[i]; new_path = items[i+1]
            i += 2
            p = new_path or old_path
        else:
            if i >= len(items):
                break
            p = items[i]
            i += 1
        if p and (is_yaml(p) or is_gradle(p)):
            paths.append(p)
    return paths

def blob_sha(repo: Path, commit: str, path: str) -> Optional[str]:
    try:
        # format: "<mode> <type> <sha>\t<path>"
        out = run_git(repo, ["ls-tree", commit, "--", path]).stdout.strip()
        if not out:
            return None
        for line in out.splitlines():
            parts = line.split()
            if len(parts) >= 3:
                sha = parts[2]
                return sha
        return None
    except Exception:
        return None

def blob_size(repo: Path, sha: str) -> int:
    try:
        return int(run_git(repo, ["cat-file", "-s", sha]).stdout.strip())
    except Exception:
        return 0

@lru_cache(maxsize=100_000)
def read_blob_cached(repo_path: str, sha: str) -> Optional[str]:
    # cache key is (repo_path, sha)
    repo = Path(repo_path)
    try:
        return run_git(repo, ["cat-file", "-p", sha]).stdout
    except Exception:
        return None

# ---------------------------
# Detection logic (source-aware)
# ---------------------------
def detect_styles_in_text(content: str, path: str) -> Set[str]:
    styles: Set[str] = set()

    for rx in GENERIC_PATTERNS_ALL:
        if rx.search(content):
            styles.add("GENERIC")
            break

    for rx in DIY_PATTERNS_ALL:
        if rx.search(content):
            styles.add("DIY")
            break

    if is_gradle(path):
        for rx in GMD_PATTERNS_GRADLE:
            if rx.search(content):
                styles.add("GMD_GRADLE")
                break

    if is_yaml(path):
        for rx in GMD_PATTERNS_YAML_INTENT:
            if rx.search(content):
                styles.add("GMD_YAML")
                break

    return styles

# ---------------------------
# Timeline (fast implementation)
# ---------------------------
def empty_result(repo: Path, cutoff_iso: str) -> Dict:
    return {
        "repo_name": repo.name,
        "repo_path": str(repo),
        "cutoff_date": cutoff_iso,
        "first_commit_date": None,
        "timeline": [],
        "events": {"DIY": [], "GENERIC": [], "GMD": [], "INTENT_GMD": []},
        "snapshot_as_of_cutoff": [],
    }

def build_timeline_fast(repo: Path, cutoff_iso: str) -> Dict:
    commits = commits_touching_relevant(repo, cutoff_iso)
    if not commits:
        return empty_result(repo, cutoff_iso)

    commit_dates: Dict[str, str] = {}
    for c in commits:
        d = get_commit_date_iso(repo, c)
        if d:
            commit_dates[c] = d
    first_commit_date = commit_dates.get(commits[0])

    prev: Set[str] = set()
    timeline: List[Dict] = []
    events: Dict[str, List[Dict]] = {"DIY": [], "GENERIC": [], "GMD": [], "INTENT_GMD": []}

    for c in commits:
        if SKIP_MERGE_COMMITS and is_merge(repo, c):
            continue

        changed = changed_relevant_files(repo, c)
        if not changed:
            continue

        seen_yaml: Set[str] = set()
        seen_gradle: Set[str] = set()

        for path in changed:
            sha = blob_sha(repo, c, path)
            if not sha:
                continue
            if BLOB_MAX_SIZE and blob_size(repo, sha) > BLOB_MAX_SIZE:
                continue

            text = read_blob_cached(str(repo), sha)
            if not text:
                continue
            if len(text) > 1_500_000:  # hard stop (safety like original)
                continue

            raw = detect_styles_in_text(text, path)
            if is_yaml(path):
                seen_yaml |= raw
            elif is_gradle(path):
                seen_gradle |= raw

        # Consolidate final styles (same semantics as original)
        final_styles: Set[str] = set()
        if "DIY" in (seen_yaml | seen_gradle):
            final_styles.add("DIY")
        if "GENERIC" in (seen_yaml | seen_gradle):
            final_styles.add("GENERIC")
        if "GMD_GRADLE" in (seen_yaml | seen_gradle):
            final_styles.add("GMD")
        elif "GMD_YAML" in (seen_yaml | seen_gradle):
            final_styles.add("INTENT_GMD")

        if final_styles != prev:
            timeline.append(
                {
                    "date": commit_dates.get(c),
                    "commit": c,
                    "styles": sorted(final_styles),
                    "sources": {
                        "yaml": sorted(seen_yaml),
                        "gradle": sorted(seen_gradle),
                    },
                }
            )
            added = final_styles - prev
            removed = prev - final_styles
            for s in sorted(added):
                events[s].append({"event": "added", "date": commit_dates.get(c), "commit": c})
            for s in sorted(removed):
                events[s].append({"event": "removed", "date": commit_dates.get(c), "commit": c})
            prev = final_styles

    snapshot = sorted(prev)
    return {
        "repo_name": repo.name,
        "repo_path": str(repo),
        "cutoff_date": cutoff_iso,
        "first_commit_date": first_commit_date,
        "timeline": timeline,
        "events": events,
        "snapshot_as_of_cutoff": snapshot,
    }

# ---------------------------
# Runner (threaded, resumable)
# ---------------------------
def normalize_cutoff(cutoff_iso: str) -> str:
    try:
        _dt = datetime.fromisoformat(cutoff_iso.replace("Z", "+00:00"))
        return _dt.isoformat()
    except Exception:
        print(f"[warn] Could not parse cutoff '{cutoff_iso}', using raw string for git", file=sys.stderr)
        return cutoff_iso

def output_path_for(repo: Path) -> Path:
    return OUTPUT_MINE / f"{repo.name}.emulator_timeline.json"

def process_one_repo(repo: Path, cutoff_iso: str) -> Tuple[Path, Optional[Dict], Optional[str]]:
    # Return (repo, data_or_none, error_or_none)
    try:
        if RESUME_IF_EXISTS:
            out_path = output_path_for(repo)
            if out_path.exists():
                return repo, None, "skipped_exists"
        data = build_timeline_fast(repo, cutoff_iso)
        return repo, data, None
    except Exception as e:
        return repo, None, f"{e}"

def run_miner_fast(cutoff_iso: str = CUTOFF_DEFAULT, max_repos: int = MAX_REPOS) -> None:
    OUTPUT_MINE.mkdir(parents=True, exist_ok=True)
    cutoff_iso = normalize_cutoff(cutoff_iso)

    repos = list_repos(INPUT_CLONES)
    if max_repos and max_repos > 0:
        repos = repos[:max_repos]

    if not repos:
        print(f"[info] No git repos found under: {INPUT_CLONES}")
        return

    print(f"[info] Base folder: {BASE_FOLDER}")
    print(f"[info] Input (repos): {INPUT_CLONES}")
    print(f"[info] Output (JSON): {OUTPUT_MINE}")
    print(f"[info] Found {len(repos)} repos")
    print(f"[info] Parallel workers (threads): {MAX_WORKERS}")
    print(f"[info] Options: skip_merges={SKIP_MERGE_COMMITS}, blob_max_size={BLOB_MAX_SIZE}, resume={RESUME_IF_EXISTS}")

    written = 0
    skipped = 0
    errors = 0
    with PoolExecutor(max_workers=MAX_WORKERS) as ex:     # THREADS, not processes
        futures = {ex.submit(process_one_repo, r, cutoff_iso): r for r in repos}
        total = len(futures)
        for i, fut in enumerate(as_completed(futures), 1):
            repo = futures[fut]
            rel = repo.relative_to(INPUT_CLONES) if repo != INPUT_CLONES else Path(repo.name)
            try:
                repo_, data, err = fut.result()
                if err == "skipped_exists":
                    print(f"[{i}/{total}] Skipped (exists): {rel}")
                    skipped += 1
                    continue
                if err:
                    print(f"[{i}/{total}] [error] {rel}: {err}", file=sys.stderr)
                    errors += 1
                    continue
                out_path = output_path_for(repo_)
                with out_path.open("w", encoding="utf-8") as f:
                    json.dump(data, f, ensure_ascii=False, indent=2)
                print(f"[{i}/{total}] Wrote: {rel}")
                written += 1
            except Exception as e:
                print(f"[{i}/{total}] [error] {rel}: {e}", file=sys.stderr)
                errors += 1

    print(f"[done] JSON files written to: {OUTPUT_MINE} (written={written}, skipped={skipped}, errors={errors})")

# ---- Execute immediately when this cell runs ----
if __name__ == "__main__":
    run_miner_fast()
else:
    # If running in a notebook cell, call directly:
    run_miner_fast()


[info] Base folder: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2
[info] Input (repos): C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones
[info] Output (JSON): C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\mine
[info] Found 299 repos
[info] Parallel workers (threads): 32
[info] Options: skip_merges=True, blob_max_size=600000, resume=True
[1/299] Wrote: arkivanov__android-dev-challenge-compose-2
[2/299] Wrote: AAkira__ExpandableLayout
[3/299] Wrote: ajitsing__ExpenseManager
[4/299] Wrote: AndProx__AndProx
[5/299] Wrote: bilibili__boxing
[6/299] Wrote: blazsolar__android-collapse-calendar-view
[7/299] Wrote: andremion__Villains-and-Heroes
[8/299] Wrote: alexbakker__webdav-provider
[9/299] Wrote: 4eRTuk__audioview
[10/299] Wrote: CarlosMChica__easyrecycleradapters
[11/299] Wrote: brarcher__video-transcoder
[12/299] Wrote: a914-gowtham__compose-ratingbar
[13/299] Wrote: AdamMc331__AndroidStudyGuide
[14/299] Wrote: AChep__AcDisplay
[15/299] Wrote: alexsty

## Step 3: labeling

## Step 4: Combine